# Attention Extraction (Qwen3-1.7B)  
### Google Colab notebook (1 of 2)

**Before running:** `Runtime > Change runtime type > Hardware accelerator > GPU` (a free **T4** is enough for Qwen3-1.7B).

To open in Colab: go to [colab.research.google.com](https://colab.research.google.com) > `File > Upload notebook`, and drop this `.ipynb` in. Run the cells top to bottom.

The first cell installs dependencies and the second mounts Google Drive so data persists between the two notebooks. Set `USE_DRIVE = False` in both if you prefer ephemeral storage.

---

In [ ]:
# --- Colab environment setup -------------------------------------------------
# Detect Colab, install the versions Qwen3 needs, and confirm a GPU is attached.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    import subprocess, sys
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '-U',
         'transformers>=4.51', 'datasets>=2.19', 'accelerate', 'pyarrow'],
        check=True,
    )

import torch
print('In Colab      :', IN_COLAB)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU           :', torch.cuda.get_device_name(0))
    print('bf16 native   :', torch.cuda.is_bf16_supported(), '(T4 = False -> fp16 is used)')
else:
    print('*** No GPU. In Colab: Runtime > Change runtime type > Hardware accelerator > GPU (T4). ***')

## 0. Storage (Google Drive)

In [ ]:
# --- Storage location --------------------------------------------------------
# Colab's local disk is wiped when the runtime resets. Mount Google Drive so the
# raw attention tensors written here survive into Notebook 2's session.
USE_DRIVE = True   # set False to use ephemeral /content storage (lost on reset)

from pathlib import Path
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/attention_sink_project')
elif IN_COLAB:
    BASE = Path('/content/attention_sink_project')
else:
    BASE = Path('.')            # running locally
BASE.mkdir(parents=True, exist_ok=True)

DATA_ROOT = str(BASE / 'attention_sink_data')   # <- ExtractConfig.out_dir uses this
print('Base path :', BASE)
print('Data root :', DATA_ROOT)

# Notebook 1 — Attention Extraction (Qwen3-1.7B)

**Role in the pipeline:** *extraction only*. This notebook loads the model,
standardises the prompts, and saves the **raw, un-averaged** attention tensors
to disk. It performs **no** aggregation, no sink metric, and no plotting — all
analysis lives in Notebook 2, which derives every statistic from the raw
tensors saved here.

### Design philosophy
> Store the most granular measurements first. Aggregate only when answering a
> specific research question.

### Methodology requirements addressed here
| # | Requirement | Where |
|---|-------------|-------|
| 6 | Parallel bilingual prompts (FLORES-200 En–Vi) | §6 |
| 7 | Verify the identity of token 0 (BOS vs first content token) | §5 |
| 9 | Preserve raw measurements (save full `[Layer, Head, Q, K]` tensors, no averaging) | §7–§9 |
| — | Modular config, logging, reproducibility, metadata, docs | throughout |

### What gets written to `out_dir/`
- `attn/<prompt_id>.npz` — raw attention `[L, H, S, S]` per prompt (ragged `S`, so one file each)
- `metadata.parquet` / `metadata.csv` — one row per prompt (language, category, seq_len, token-0 info, …)
- `token_ids.json` — token ids per prompt
- `config.json` — the exact run configuration (reproducibility)
- `extraction.log` — full run log


## 1. Environment & imports

In [ ]:
import os, sys, json, time, random, logging
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Optional

import numpy as np
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM

print('Python      :', sys.version.split()[0])
print('NumPy       :', np.__version__)
print('PyTorch     :', torch.__version__)
print('Transformers:', transformers.__version__)
print('CUDA        :', torch.cuda.is_available(),
      ('| ' + torch.cuda.get_device_name(0)) if torch.cuda.is_available() else '')

## 2. Configuration (modular)

Everything that can change lives in one dataclass. Nothing below reads a magic
constant. The `prompt_mode` field is the knob for **token-0 standardisation**
(Requirement 7):

- `prepend_bos`  — prepend a single fixed token so token 0 is **identical** for every prompt (default; the most defensible choice for cross-prompt sink comparison).
- `chat_template` — wrap each prompt in the Qwen chat template (token 0 is always `<|im_start|>`).
- `raw` — tokenizer defaults; Qwen may **not** add a BOS token, so token 0 becomes the first *content* token and can differ across prompts. Kept available so the limitation can be demonstrated.

In [ ]:
@dataclass
class ExtractConfig:
    # --- Model ---
    model_name: str = 'Qwen/Qwen3-1.7B'
    attn_implementation: str = 'eager'   # REQUIRED: sdpa/flash do NOT return attention weights
    torch_dtype: str = 'auto'            # 'auto' -> bf16 on GPU, fp32 on CPU
    device_map: Optional[str] = None     # 'auto' for multi-GPU sharding; None -> single device

    # --- Token-0 standardisation (Requirement 7) ---
    prompt_mode: str = 'prepend_bos'     # 'prepend_bos' | 'chat_template' | 'raw'
    fixed_prefix_token: Optional[str] = None  # None -> auto (bos else eos)

    # --- Parallel bilingual data (Requirement 6) ---
    src_lang: str = 'eng_Latn'
    tgt_lang: str = 'vie_Latn'
    flores_split: str = 'dev'
    n_pairs: int = 20                    # sentence pairs -> n_pairs En + n_pairs Vi prompts
    max_tokens: int = 96                 # drop pathologically long sentences
    random_seed: int = 20240517

    # --- Storage (Requirement 9) ---
    out_dir: str = DATA_ROOT
    store_dtype: str = 'float16'         # on-disk dtype for raw attention (probs in [0,1])

cfg = ExtractConfig()
print(json.dumps(asdict(cfg), indent=2))

## 3. Reproducibility & logging

In [ ]:
def set_reproducibility(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    transformers.set_seed(seed)
    # Attention read-out is deterministic (no sampling); seed fixes data subsampling order.

def get_logger(name='extract', log_file=None):
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    fmt = logging.Formatter('%(asctime)s | %(levelname)-7s | %(message)s', '%H:%M:%S')
    sh = logging.StreamHandler(sys.stdout); sh.setFormatter(fmt); logger.addHandler(sh)
    if log_file:
        fh = logging.FileHandler(log_file, mode='w'); fh.setFormatter(fmt); logger.addHandler(fh)
    logger.propagate = False
    return logger

set_reproducibility(cfg.random_seed)
OUT = Path(cfg.out_dir)
(OUT / 'attn').mkdir(parents=True, exist_ok=True)
logger = get_logger(log_file=str(OUT / 'extraction.log'))
logger.info('Reproducibility set (seed=%d)', cfg.random_seed)

## 4. Load tokenizer & model

`attn_implementation='eager'` is **mandatory** — the fused SDPA / FlashAttention
kernels never materialise the attention matrix, so `output_attentions=True`
returns `None` with them. The number of layers/heads is read from the model,
not hard-coded (Qwen3-1.7B: 28 layers, 16 query heads).

In [ ]:
def resolve_dtype(name):
    if name == 'auto':
        if torch.cuda.is_available():
            return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        return torch.float32
    return getattr(torch, name)

logger.info('Loading tokenizer: %s', cfg.model_name)
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)

logger.info('Loading model (attn_implementation=%s) ...', cfg.attn_implementation)
model = AutoModelForCausalLM.from_pretrained(
    cfg.model_name,
    torch_dtype=resolve_dtype(cfg.torch_dtype),
    attn_implementation=cfg.attn_implementation,
    device_map=cfg.device_map,
)
if cfg.device_map is None:
    model = model.to('cuda' if torch.cuda.is_available() else 'cpu')
model.eval()

NUM_LAYERS = model.config.num_hidden_layers
NUM_HEADS  = model.config.num_attention_heads
logger.info('Loaded. layers=%d  query_heads=%d  kv_heads=%s  device=%s',
            NUM_LAYERS, NUM_HEADS,
            getattr(model.config, 'num_key_value_heads', 'n/a'), model.device)

## 5. Verify the identity of token 0 (Requirement 7)

We do **not** assume token 0 is a BOS token. Below we (a) inspect the
tokenizer's special-token behaviour, (b) build inputs under the configured
`prompt_mode`, and (c) confirm that token 0 has the **same** semantic role
across English *and* Vietnamese prompts. With `prepend_bos` this is guaranteed
by construction; the check still runs and is logged so the assumption is
*verified automatically rather than assumed*.

In [ ]:
def choose_prefix_token_id(tokenizer, cfg):
    if cfg.fixed_prefix_token is not None:
        return tokenizer.convert_tokens_to_ids(cfg.fixed_prefix_token)
    if tokenizer.bos_token_id is not None:
        return tokenizer.bos_token_id
    return tokenizer.eos_token_id   # Qwen: <|endoftext|> acts as a stable fixed prefix

PREFIX_ID = choose_prefix_token_id(tokenizer, cfg)

def build_inputs(text, cfg, tokenizer):
    '''Return (input_ids[1,S], attention_mask[1,S], token0_info) for one prompt.'''
    if cfg.prompt_mode == 'chat_template':
        input_ids = tokenizer.apply_chat_template(
            [{'role': 'user', 'content': text}],
            add_generation_prompt=True, return_tensors='pt')
    elif cfg.prompt_mode == 'prepend_bos':
        body = tokenizer(text, add_special_tokens=False, return_tensors='pt')['input_ids']
        prefix = torch.tensor([[PREFIX_ID]], dtype=body.dtype)
        input_ids = torch.cat([prefix, body], dim=1)
    elif cfg.prompt_mode == 'raw':
        input_ids = tokenizer(text, return_tensors='pt')['input_ids']
    else:
        raise ValueError('unknown prompt_mode: ' + cfg.prompt_mode)
    attention_mask = torch.ones_like(input_ids)
    t0 = int(input_ids[0, 0])
    info = {
        'prompt_mode': cfg.prompt_mode,
        'token0_id': t0,
        'token0_str': tokenizer.convert_ids_to_tokens([t0])[0],
        'token0_is_bos': (tokenizer.bos_token_id is not None and t0 == tokenizer.bos_token_id),
        'seq_len': int(input_ids.shape[1]),
    }
    return input_ids, attention_mask, info

print('bos_token     :', tokenizer.bos_token, '| id:', tokenizer.bos_token_id)
print('eos_token     :', tokenizer.eos_token, '| id:', tokenizer.eos_token_id)
print('add_bos_token :', getattr(tokenizer, 'add_bos_token', 'n/a'))
print('chosen prefix :', tokenizer.convert_ids_to_tokens([PREFIX_ID])[0], '(id', PREFIX_ID, ')')
print('prompt_mode   :', cfg.prompt_mode)
print('-' * 66)
_probe = ['The cat sat on the mat.', 'Con meo ngoi tren tam tham.']
_t0_ids = set()
for s in _probe:
    _, _, info = build_inputs(s, cfg, tokenizer)
    _t0_ids.add(info['token0_id'])
    print('token0_id=%-7d token0=%-14r is_bos=%s seq_len=%-3d | %s'
          % (info['token0_id'], info['token0_str'], info['token0_is_bos'], info['seq_len'], s))
print('-' * 66)
print('token 0 identical across sampled languages :', len(_t0_ids) == 1,
      '' if len(_t0_ids) == 1 else '(EXPECTED with prompt_mode=raw; token 0 is first content token)')

## 6. Parallel bilingual prompts (Requirement 6)

Prompts are **sentence-aligned** pairs so language is the only thing that varies
within a pair — semantic content is held constant. We prefer **FLORES-200**
(`eng_Latn` / `vie_Latn`); if the dataset can't be downloaded we fall back to a
small embedded parallel corpus (with category labels) so the pipeline is fully
runnable offline. Every English prompt has a direct Vietnamese translation and
they share a `pair_id` — each pair is one matched experimental unit.

In [ ]:
# Embedded fallback: sentence-aligned En-Vi pairs across categories.
EMBEDDED_PAIRS = [
    {'category':'news',          'eng':'The government announced a new policy to reduce carbon emissions by 2030.', 'vie':'Chinh phu da cong bo mot chinh sach moi nham giam luong khi thai carbon vao nam 2030.'},
    {'category':'news',          'eng':'Scientists discovered a new species of fish in the deep ocean.',            'vie':'Cac nha khoa hoc da phat hien mot loai ca moi o vung bien sau.'},
    {'category':'news',          'eng':'The central bank decided to keep interest rates unchanged this month.',     'vie':'Ngan hang trung uong da quyet dinh giu nguyen lai suat trong thang nay.'},
    {'category':'conversational','eng':'Could you please tell me how to get to the train station?',                 'vie':'Ban co the vui long chi cho toi duong den ga tau khong?'},
    {'category':'conversational','eng':'I really enjoyed the dinner we had last night.',                            'vie':'Toi thuc su rat thich bua toi ma chung ta da an toi qua.'},
    {'category':'conversational','eng':'What time does the meeting start tomorrow morning?',                        'vie':'Cuoc hop bat dau luc may gio vao sang mai?'},
    {'category':'technical',     'eng':'The algorithm sorts the array in logarithmic time on average.',            'vie':'Thuat toan sap xep mang trong thoi gian trung binh theo ham logarit.'},
    {'category':'technical',     'eng':'Increase the sample size to reduce the variance of the estimate.',          'vie':'Hay tang kich thuoc mau de giam phuong sai cua uoc luong.'},
    {'category':'technical',     'eng':'The model was trained on a large dataset of labeled images.',               'vie':'Mo hinh duoc huan luyen tren mot tap du lieu lon gom cac hinh anh da duoc gan nhan.'},
    {'category':'literary',      'eng':'The old house stood silently at the edge of the forest.',                   'vie':'Ngoi nha cu dung lang le o bia rung.'},
    {'category':'literary',      'eng':'She watched the rain fall gently on the quiet street.',                     'vie':'Co ngam nhin mua roi nhe nhang tren con pho vang lang.'},
    {'category':'literary',      'eng':'A cold wind swept across the empty fields at dawn.',                        'vie':'Mot con gio lanh quet qua nhung canh dong trong vao luc binh minh.'},
    {'category':'everyday',      'eng':'Please remember to buy milk and eggs on your way home.',                    'vie':'Nho mua sua va trung tren duong ve nha nhe.'},
    {'category':'everyday',      'eng':'The children played happily in the park all afternoon.',                    'vie':'Bon tre choi dua vui ve trong cong vien suot buoi chieu.'},
    {'category':'everyday',      'eng':'He fixed the broken bicycle in less than an hour.',                         'vie':'Anh ay da sua xong chiec xe dap hong trong chua day mot gio.'},
]

def load_parallel_pairs(cfg, tokenizer, logger):
    pairs = None
    try:
        from datasets import load_dataset
        try:  # newer paired config
            ds = load_dataset('facebook/flores', cfg.src_lang + '-' + cfg.tgt_lang, split=cfg.flores_split)
            sc, tc = 'sentence_' + cfg.src_lang, 'sentence_' + cfg.tgt_lang
            pairs = [{'category':'flores', 'eng':r[sc], 'vie':r[tc]} for r in ds]
        except Exception:  # per-language configs aligned by row index
            src = load_dataset('facebook/flores', cfg.src_lang, split=cfg.flores_split)
            tgt = load_dataset('facebook/flores', cfg.tgt_lang, split=cfg.flores_split)
            pairs = [{'category':'flores', 'eng':s['sentence'], 'vie':t['sentence']}
                     for s, t in zip(src, tgt)]
        logger.info('Loaded %d FLORES-200 pairs', len(pairs))
    except Exception as e:
        logger.warning('FLORES-200 unavailable (%s). Falling back to embedded corpus.', type(e).__name__)
        pairs = list(EMBEDDED_PAIRS)

    rng = random.Random(cfg.random_seed)
    rng.shuffle(pairs)
    kept = []
    for p in pairs:
        le = len(tokenizer(p['eng'], add_special_tokens=False)['input_ids'])
        lv = len(tokenizer(p['vie'], add_special_tokens=False)['input_ids'])
        if le <= cfg.max_tokens and lv <= cfg.max_tokens:
            kept.append(p)
        if len(kept) >= cfg.n_pairs:
            break
    logger.info('Using %d pairs after length filtering (max_tokens=%d)', len(kept), cfg.max_tokens)
    return kept

pairs = load_parallel_pairs(cfg, tokenizer, logger)

# Expand each pair into two matched prompts sharing pair_id.
prompts = []
for i, p in enumerate(pairs):
    prompts.append({'prompt_id': '%03d_en' % i, 'pair_id': i, 'language':'en', 'category':p['category'], 'text':p['eng']})
    prompts.append({'prompt_id': '%03d_vi' % i, 'pair_id': i, 'language':'vi', 'category':p['category'], 'text':p['vie']})
print('%d prompts (%d pairs x 2 languages)' % (len(prompts), len(pairs)))

## 7. Helper functions — attention extraction (no averaging)

`extract_attention` returns the **full** `[L, H, S, S]` attention tensor for one
prompt. No averaging over heads, layers, or query positions happens anywhere in
this notebook.

In [ ]:
@torch.no_grad()
def extract_attention(text, cfg, tokenizer, model):
    input_ids, attention_mask, info = build_inputs(text, cfg, tokenizer)
    input_ids = input_ids.to(model.device)
    attention_mask = attention_mask.to(model.device)
    out = model(input_ids=input_ids, attention_mask=attention_mask,
                output_attentions=True, use_cache=False)
    # out.attentions: tuple(L) of [1, H, S, S]
    attn = torch.stack(out.attentions, dim=0)[:, 0]        # [L, H, S, S], drop batch
    attn = attn.to(torch.float32).cpu().numpy()            # promote before numpy
    return attn, input_ids[0].cpu().numpy(), info

def save_prompt(rec, attn, token_ids, cfg):
    store = getattr(np, cfg.store_dtype)
    np.savez_compressed(
        Path(cfg.out_dir) / 'attn' / (rec['prompt_id'] + '.npz'),
        attn=attn.astype(store),
        token_ids=np.asarray(token_ids, dtype=np.int64),
    )

## 8. Run extraction

In [ ]:
import pandas as pd

meta_rows = []
t_start = time.time()
for k, rec in enumerate(prompts):
    attn, token_ids, info = extract_attention(rec['text'], cfg, tokenizer, model)  # [L,H,S,S] RAW
    save_prompt(rec, attn, token_ids, cfg)

    row = dict(rec)
    row.update({
        'seq_len':       info['seq_len'],
        'n_layers':      int(attn.shape[0]),
        'n_heads':       int(attn.shape[1]),
        'token0_id':     info['token0_id'],
        'token0_str':    info['token0_str'],
        'token0_is_bos': info['token0_is_bos'],
        'prompt_mode':   info['prompt_mode'],
        'attn_file':     'attn/' + rec['prompt_id'] + '.npz',
        'token_ids':     token_ids.tolist(),
        # integrity check: each causal attention row must sum to ~1
        'row_sum_ok':    bool(np.allclose(attn.sum(-1), 1.0, atol=1e-2)),
    })
    meta_rows.append(row)
    if k % 5 == 0 or k == len(prompts) - 1:
        logger.info('[%2d/%2d] %s  seq_len=%-3d  attn%s  row_sum_ok=%s',
                    k + 1, len(prompts), rec['prompt_id'], info['seq_len'],
                    tuple(attn.shape), row['row_sum_ok'])

logger.info('Extraction finished in %.1fs', time.time() - t_start)

## 9. Save metadata & config (reproducibility)

In [ ]:
meta = pd.DataFrame(meta_rows)

# Metadata table (token_ids kept out of the flat CSV; stored separately as JSON).
try:
    meta.to_parquet(OUT / 'metadata.parquet')
    logger.info('Wrote metadata.parquet')
except Exception as e:
    logger.warning('parquet unavailable (%s); CSV only', type(e).__name__)
meta.drop(columns=['token_ids']).to_csv(OUT / 'metadata.csv', index=False)

with open(OUT / 'token_ids.json', 'w') as f:
    json.dump({r['prompt_id']: r['token_ids'] for r in meta_rows}, f)
with open(OUT / 'config.json', 'w') as f:
    json.dump(asdict(cfg), f, indent=2)

logger.info('Saved metadata.csv, token_ids.json, config.json to %s', OUT)
meta.drop(columns=['token_ids']).head()

## 10. Sanity checks (no analysis)

In [ ]:
# Verify shapes, causal masking, and that token 0 is standardised. No metrics here.
n_files = len(list((OUT / 'attn').glob('*.npz')))
print('attention files on disk :', n_files, '(expected', len(prompts), ')')
print('all rows sum to ~1      :', bool(meta['row_sum_ok'].all()))
print('unique token0_id        :', sorted(meta['token0_id'].unique()),
      '<- single value => token 0 standardised' if meta['token0_id'].nunique() == 1 else '')
print('seq_len by language     :')
print(meta.groupby('language')['seq_len'].describe()[['count','mean','min','max']])

# Spot-check one saved tensor + confirm causal structure (upper triangle is zero).
_d = np.load(OUT / 'attn' / (prompts[0]['prompt_id'] + '.npz'))
_a = _d['attn'].astype(np.float32)
print('\nsample tensor', prompts[0]['prompt_id'], 'shape', _a.shape)
print('upper-triangle mass (should be ~0):', float(np.triu(_a[0, 0], k=1).sum()))

---
### Done — hand-off to Notebook 2
Raw attention tensors and metadata are on disk. Notebook 2 loads them and
derives **every** statistic (Layer×Head sink matrix, layer progression,
sequence-length analysis, …) from these raw tensors. Nothing was averaged here.

## Download results

In [ ]:
# --- Persist / download the raw dataset --------------------------------------
if IN_COLAB and USE_DRIVE:
    print('Raw attention + metadata saved to Google Drive at:', DATA_ROOT)
    print('Open Notebook 2 (with the same USE_DRIVE setting) to analyse it.')
elif IN_COLAB:
    import shutil
    from google.colab import files
    zp = shutil.make_archive('/content/attention_sink_data', 'zip', DATA_ROOT)
    print('Ephemeral storage -> zipping for download:', zp)
    files.download(zp)
else:
    print('Saved under:', DATA_ROOT)